# 06 · 統計檢定：這個差異是真的，還是運氣？

EDA 看到「女性生還率比男性高」,但這會不會只是**抽樣的運氣**?統計檢定就是用來回答這個問題的:

> **假設檢定**先假設「其實沒差」(虛無假設),再算出「如果真的沒差,卻看到這麼大的差異」的機率——這就是 **p 值**。p 值很小(慣例 < 0.05),就有信心說「差異是真的」。

這也是 **A/B test** 的數學基礎:網站改版有沒有真的提升轉換率,用的就是同一套。

## 1. 載入乾淨資料

In [ ]:
%pip install -q -U seaborn scikit-learn

In [ ]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")
df["age"] = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.drop(columns=["deck"])
print("乾淨資料:", df.shape)

## 2. t 檢定:生還者與罹難者的票價有差嗎?

比較兩組**數值**的平均,用獨立樣本 t 檢定。

In [ ]:
from scipy import stats

fare_survived = df[df["survived"] == 1]["fare"]
fare_died = df[df["survived"] == 0]["fare"]
t, p = stats.ttest_ind(fare_survived, fare_died, equal_var=False)
print(f"生還者平均票價 {fare_survived.mean():.1f} vs 罹難者 {fare_died.mean():.1f}")
print(f"t = {t:.2f},  p = {p:.2e}")
print("→ p 遠小於 0.05:票價差異是真的,不是運氣。")

## 3. 卡方檢定:性別與生還有關聯嗎?

兩個**類別**變數有沒有關聯,用卡方檢定(看交叉表的實際 vs 期望)。

In [ ]:
table = pd.crosstab(df["sex"], df["survived"])
print("交叉表:"); print(table)
chi2, p, dof, _ = stats.chi2_contingency(table)
print(f"\nchi2 = {chi2:.1f},  p = {p:.2e}")
print("→ p 極小:性別與生還高度相關,絕非巧合。")

## 4. p 值的正確心態

p 值小 = 「差異不太可能只是運氣」,**不等於**「差異很大」或「很重要」。樣本夠大時,再小的差異 p 值也會很小。**統計顯著 ≠ 實務重要**,兩者要分開看。

In [ ]:
print("檢定告訴你『有沒有差』(顯著性);效果大小告訴你『差多少』(重要性)。")
print("好的分析兩個都報:p 值 + 實際差距。")

## 小結

- **假設檢定 + p 值**:判斷觀察到的差異是真的,還是抽樣運氣。
- **t 檢定**比兩組數值平均(票價);**卡方檢定**看兩個類別的關聯(性別 vs 生還)。
- **統計顯著 ≠ 實務重要**:p 值要搭配效果大小一起解讀。這正是 A/B test 的核心邏輯。

下一課:把分析收斂成一個**預測模型**,接上 `ml/scikit-learn`。